<a href="https://colab.research.google.com/github/markjschmidt/PET_MIP/blob/main/MJS_Sonnet4_PET_MIP_S51320_20261026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PET MIP Rotating Viewer — S51320

NAC 1 mm voxel Randoms Coincidence events — 164 slices (I10–I1640), ~896 KB/slice  
**Reconstruction:** NAC (no attenuation correction), 1 mm voxels  
**Path:** `DICOM/S51320/`  
> Orientation, units and spacing are auto-detected from DICOM headers after LFS pull (Step 3).

## Step 1 — Install dependencies

In [ ]:
!apt-get install -y git-lfs -q
!pip install pydicom scipy ipywidgets -q
from google.colab import output
output.enable_custom_widget_manager()

## Step 2 — Clone repo and pull DICOM files from Git LFS

In [ ]:
import os, glob

SERIES_NAME = 'S51320'
REPO_URL  = 'https://github.com/markjschmidt/PET_MIP'
CLONE_DIR = '/content/PET_MIP'
DICOM_DIR = f'{CLONE_DIR}/DICOM/S51320'

if not os.path.isdir(CLONE_DIR):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}
    !git -C {CLONE_DIR} lfs pull --include='DICOM/S51320/*'
else:
    print('Repo already cloned — skipping.')

# Use I* glob to exclude DIRFILE and dirty non-DICOM files
files = sorted(glob.glob(f'{DICOM_DIR}/I*'))
print(f'Found {len(files)} DICOM files.')

sizes = [os.path.getsize(f) for f in files[:3]]
if all(s < 500 for s in sizes):
    print('WARNING: LFS pointer stubs detected — LFS pull may have failed.')
else:
    print(f'LFS confirmed — first slice: {sizes[0]:,} bytes')

## Step 3 — Build 3D volume
Auto-detects slice orientation from `ImageOrientationPatient`, sorts slices along the correct patient axis, and applies DICOM rescale.

In [ ]:
import pydicom
import numpy as np

# Read orientation from first slice to determine stacking axis
ds_hdr  = pydicom.dcmread(files[0], stop_before_pixels=True)
iop     = [float(x) for x in ds_hdr.ImageOrientationPatient]
normal  = np.cross(iop[:3], iop[3:])          # normal to image plane
slice_axis = int(np.argmax(np.abs(normal)))    # 0=sagittal 1=coronal 2=axial
orient_names = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}
print(f'Detected orientation: {orient_names[slice_axis]}  (slice_axis={slice_axis})')

# Sort slices by position along the slice axis
slice_list = []
for f in files:
    ds  = pydicom.dcmread(f, stop_before_pixels=True)
    pos = float(ds.ImagePositionPatient[slice_axis])
    slice_list.append((pos, f))
slice_list.sort(key=lambda x: x[0])

# Metadata from first slice
ds0        = pydicom.dcmread(slice_list[0][1], stop_before_pixels=True)
units      = getattr(ds0, 'Units',             'unknown')
px_spacing = getattr(ds0, 'PixelSpacing',       [2.0, 2.0])
rows       = getattr(ds0, 'Rows',              None)
cols       = getattr(ds0, 'Columns',           None)
series_desc= getattr(ds0, 'SeriesDescription', SERIES_NAME)

row_spacing = float(px_spacing[0])   # mm per image row
col_spacing = float(px_spacing[1])   # mm per image col
slice_spacing = abs(slice_list[1][0] - slice_list[0][0]) if len(slice_list) > 1 else row_spacing

print(f'Series       : {series_desc}')
print(f'Units        : {units}')
print(f'Matrix       : {rows} x {cols}')
print(f'Row spacing  : {row_spacing} mm  |  Col spacing: {col_spacing} mm')
print(f'Slice spacing: {slice_spacing:.3f} mm')
print(f'Position range: {slice_list[0][0]:.1f} → {slice_list[-1][0]:.1f} mm')

# Build volume — shape depends on orientation
volume = []
for _, f in slice_list:
    ds     = pydicom.dcmread(f)
    arr    = ds.pixel_array.astype(np.float32)
    slope  = float(getattr(ds, 'RescaleSlope',     1))
    intcpt = float(getattr(ds, 'RescaleIntercept', 0))
    volume.append(arr * slope + intcpt)

vol = np.array(volume)
# vol.shape = (N_slices, rows, cols)
# For coronal: (Y, Z, X); for axial: (Z, Y, X); for sagittal: (X, Z, Y)
print(f'\nVolume shape : {vol.shape}  (slices, rows, cols)')
print(f'Pixel range  : {vol.min():.1f} → {vol.max():.1f} {units}')

## Step 3b — Interactive slice viewer
Left panel: native acquired slices.  Right panel: cross-section perpendicular to the slice direction.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display

n_slices, n_rows, n_cols = vol.shape

# 95th percentile avoids saturation on uniform phantom / randoms data
SLICE_VMAX = float(np.percentile(vol[vol > 0], 95))

# ── Aspect ratios ─────────────────────────────────────────────────────────
# native slice: rows × cols
native_aspect = row_spacing / col_spacing   # physical proportions

# Cross-section: slice_spacing (stacking dim) × col_spacing (col dim)
cross_aspect  = slice_spacing / col_spacing

# ── Sliders ───────────────────────────────────────────────────────────────
sli_slider = widgets.IntSlider(
    value=n_slices // 2, min=0, max=n_slices - 1, step=1,
    description='Slice:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'), continuous_update=True)

cross_slider = widgets.IntSlider(
    value=n_rows // 2, min=0, max=n_rows - 1, step=1,
    description='Row cross:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'), continuous_update=True)

vmax_s = widgets.FloatSlider(
    value=SLICE_VMAX,
    min=max(1.0, float(vol[vol > 0].min())),
    max=float(vol.max()),
    step=max(1.0, SLICE_VMAX / 100),
    description='Color max:',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.0f', continuous_update=False)

out_s = widgets.Output()

orient_label = {0: 'Sagittal', 1: 'Coronal', 2: 'Axial'}[slice_axis]

def render_slices(s_idx, r_idx, vmax):
    native_img = vol[s_idx, :, :]     # (rows, cols)
    cross_img  = vol[:, r_idx, :]     # (slices, cols)

    # Axial flips: anterior up, patient left on viewer's right
    if slice_axis == 2:   # axial native slice
        native_img = np.fliplr(np.flipud(native_img))
    else:
        native_img = np.flipud(native_img)

    cross_img = np.fliplr(np.flipud(cross_img))

    s_mm = slice_list[s_idx][0]

    with out_s:
        out_s.clear_output(wait=True)
        fig = plt.figure(figsize=(14, 5), facecolor='black')
        gs  = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.05], figure=fig)
        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1])
        axc = fig.add_subplot(gs[2])
        fig.patch.set_facecolor('black')
        kw = dict(cmap='hot', vmin=0, vmax=vmax, interpolation='none')

        ax1.set_facecolor('black')
        im = ax1.imshow(native_img, aspect=native_aspect, **kw)
        ax1.set_title(f'{orient_label} slice  pos={s_mm:.1f} mm  ({s_idx}/{n_slices-1})',
                      color='white', fontsize=11)
        ax1.tick_params(colors='#555')
        for sp in ax1.spines.values(): sp.set_edgecolor('#333')

        ax2.set_facecolor('black')
        ax2.imshow(cross_img, aspect=cross_aspect, **kw)
        ax2.set_title(f'Cross-section  row {r_idx}/{n_rows-1}'
                      f'  (slice_sp={slice_spacing:.2f} mm)',
                      color='white', fontsize=11)
        ax2.set_xlabel('← Pt Right   Pt Left →', color='#888', fontsize=9)
        ax2.set_ylabel('Posterior ↑ Anterior', color='#888', fontsize=9)
        ax2.tick_params(colors='#555')
        for sp in ax2.spines.values(): sp.set_edgecolor('#333')

        axc.set_facecolor('black')
        cbar = fig.colorbar(im, cax=axc)
        cbar.set_label(units, color='white', fontsize=9)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)
        fig.subplots_adjust(left=0.07, right=0.95, wspace=0.35)
        plt.show()

widgets.interactive_output(
    render_slices, {'s_idx': sli_slider, 'r_idx': cross_slider, 'vmax': vmax_s})
display(widgets.VBox([sli_slider, cross_slider,
                      widgets.HBox([vmax_s, widgets.Label(value=units)])]),
        out_s)
render_slices(n_slices // 2, n_rows // 2, SLICE_VMAX)

## Step 4 — Pre-compute rotating MIPs
Resamples the stacking axis to isotropic voxels, then rotates around the **vertical (Z) axis** at every 5°.  ~2–4 min once; slider is instant after.

In [ ]:
from scipy.ndimage import rotate, zoom
import time

# ── Rotation and projection axes depend on detected orientation ───────────
# For all orientations we want a rotating MIP around the patient Z (sup-inf) axis.
# vol.shape = (N_slices, rows, cols)
#
# Coronal (slice_axis=1): vol=(Y,Z,X), stack=Y
#   → resample Y, rotate axes=(0,2), project axis=0 → (Z,X)
# Axial (slice_axis=2): vol=(Z,Y,X), stack=Z
#   → resample Z (if needed), rotate axes=(1,2), project axis=2 → (Z,Y)
# Sagittal (slice_axis=0): vol=(X,Z,Y), stack=X
#   → resample X, rotate axes=(0,2), project axis=0 → (Z,Y)

if slice_axis == 2:   # Axial
    resample_axis  = 0
    iso_ref        = col_spacing
    rot_axes       = (1, 2)
    proj_axis      = 2
elif slice_axis == 1: # Coronal
    resample_axis  = 0
    iso_ref        = col_spacing
    rot_axes       = (0, 2)
    proj_axis      = 0
else:                 # Sagittal
    resample_axis  = 0
    iso_ref        = col_spacing
    rot_axes       = (0, 2)
    proj_axis      = 0

# Build zoom factors — only resample the stacking axis if needed
zoom_factors    = [1.0, 1.0, 1.0]
stack_spacing   = slice_spacing
zoom_factors[resample_axis] = stack_spacing / iso_ref

if abs(zoom_factors[resample_axis] - 1.0) > 0.01:
    print(f'Resampling stacking axis {resample_axis}: '
          f'zoom={zoom_factors[resample_axis]:.4f}  '
          f'({stack_spacing:.3f} mm → {iso_ref:.3f} mm)')
    vol_iso = zoom(vol, zoom_factors, order=1)
else:
    vol_iso = vol
print(f'Isotropic volume shape: {vol_iso.shape}')

# ── Pre-compute MIPs ──────────────────────────────────────────────────────
ANGLE_STEP = 5
angles     = np.arange(0, 360, ANGLE_STEP)

# Projection output shape: remove the proj_axis dimension
out_shape = list(vol_iso.shape)
out_shape.pop(proj_axis)
mip_stack = np.zeros((len(angles),) + tuple(out_shape), dtype=np.float32)

t0 = time.time()
for i, angle in enumerate(angles):
    rot          = rotate(vol_iso, angle, axes=rot_axes,
                          reshape=False, order=1, prefilter=False)
    mip_stack[i] = np.max(rot, axis=proj_axis)
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{len(angles)} angles  ({time.time()-t0:.0f}s elapsed)')

print(f'Done. {len(angles)} projections in {time.time()-t0:.0f}s.')
VMAX_DEFAULT = float(np.percentile(vol[vol > 0], 99))
print(f'MIP projection shape: {mip_stack[0].shape}')
print(f'Suggested color max (99th pct): {VMAX_DEFAULT:.0f} {units}')

## Step 5 — Interactive rotating MIP viewer

In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

angle_slider = widgets.IntSlider(
    value=0, min=0, max=355, step=ANGLE_STEP,
    description='Angle (°):',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    continuous_update=True)

vmax_slider = widgets.FloatSlider(
    value=VMAX_DEFAULT,
    min=max(1.0, float(vol[vol > 0].min())),
    max=float(vol.max()),
    step=max(1.0, VMAX_DEFAULT / 100),
    description='Color max:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='500px'),
    readout_format='.0f', continuous_update=False)

out_m = widgets.Output()

def render_mip(angle, vmax):
    idx = int(round(angle / ANGLE_STEP)) % len(angles)
    img = np.flipud(mip_stack[idx])
    with out_m:
        out_m.clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 5), facecolor='black')
        ax.set_facecolor('black')
        im = ax.imshow(img, cmap='hot', vmin=0, vmax=vmax,
                       aspect='equal', interpolation='none')
        ax.set_title(f'PET MIP  {SERIES_NAME}  |  {angle}°  |  {vmax:.0f} {units}',
                     color='white', fontsize=12, pad=8)
        ax.set_xlabel('← Pt Right   Pt Left →', color='#888', fontsize=9)
        ax.set_ylabel('Inferior ↑ Superior', color='#888', fontsize=9)
        ax.tick_params(colors='#555')
        for sp in ax.spines.values(): sp.set_edgecolor('#333')
        cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label(units, color='white', fontsize=9)
        cbar.ax.yaxis.set_tick_params(color='white')
        plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=8)
        plt.tight_layout()
        plt.show()

widgets.interactive_output(render_mip, {'angle': angle_slider, 'vmax': vmax_slider})
display(widgets.VBox([angle_slider,
                      widgets.HBox([vmax_slider, widgets.Label(value=units)])]),
        out_m)
render_mip(0, VMAX_DEFAULT)

## (Optional) Export rotating MIP as animated GIF

In [ ]:
from PIL import Image as PILImage
import io

VMAX_GIF = vmax_slider.value
frames   = []
for i in range(len(angles)):
    img = np.flipud(mip_stack[i])
    fig, ax = plt.subplots(figsize=(10, 5), facecolor='black')
    ax.set_facecolor('black')
    ax.imshow(img, cmap='hot', vmin=0, vmax=VMAX_GIF,
              aspect='equal', interpolation='none')
    ax.set_title(f'PET MIP  {SERIES_NAME}  {angles[i]:.0f}°',
                 color='white', fontsize=10)
    ax.axis('off')
    plt.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=100, facecolor='black')
    plt.close(fig)
    buf.seek(0)
    frames.append(PILImage.open(buf).convert('P', palette=PILImage.ADAPTIVE))

gif_path = f'/content/PET_MIP_{SERIES_NAME}.gif'
frames[0].save(gif_path, save_all=True, append_images=frames[1:],
               loop=0, duration=80)
print(f'Saved: {gif_path}')
from google.colab import files
files.download(gif_path)